# Notebook 1: Fitting a line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/centre-for-humanities-computing/introduction-bayesian-statistics/blob/master/notebooks/01_fitting_a_line.ipynb)

**Session 1 - Linear regressions** · Block 1

In this notebook you will:

1. look at a simulated dataset of reading times,
2. try to find the best line **by hand**,
3. let `statsmodels` find the least-squares line,
4. see what happens with more data or more noise,
5. use the line to make a prediction.

> **How to use this notebook.** Run the cells from top to bottom with **Shift + Enter**.
> Cells marked ✏️ ask you to complete or change something. If something breaks,
> use *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).

In [ ]:
# @title ⚙️ Setup: run this cell first (Shift + Enter)
import importlib.util
import subprocess
import sys
import warnings

PACKAGES = ["statsmodels", "ipywidgets"]
missing = [p for p in PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("Installing", *missing, "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Plot colours
BLUE, ROSE, GREEN, GREY = "#5B7FA6", "#BC6D86", "#5E8C4F", "#595959"
plt.rcParams.update(
    {
        "figure.figsize": (7, 4.5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#EDEDED",
        "font.size": 12,
    }
)
print("Ready!")

## 1. The data

We simulate **40 texts**. For each text we know its length (in hundreds of words, so `words = 5`
means 500 words) and how long a participant took to read it carefully, before answering
comprehension questions (in minutes).

Because we create the data ourselves, we know the **true** line:
<span style="color:#5E8C4F">intercept = 0.5, slope = 0.8, σ = 1.0</span>.
In real research you never know these values: the whole point of regression is to estimate them.

> **A simplification.** The model adds Normal noise to the line, so it can produce impossible
> negative reading times: for a 100-word text the line predicts 1.3 minutes, and with σ = 1 a
> negative time comes up about 10% of the time. Our 40 texts happen to have none, but larger
> simulated datasets later on contain a few. Models for strictly positive outcomes (for example
> on the log scale) avoid this; for now, a straight line with Normal noise is good enough.

🎬 **How these data come about.** Each text's reading time is the true line (green) plus a random error drawn from a bell curve. From the points alone, least squares (blue) finds a line close to the truth, but not exactly on it.

![Animation: points drawn from bell curves around the true line, then the least-squares line](https://raw.githubusercontent.com/centre-for-humanities-computing/introduction-bayesian-statistics/master/animations/renders/01_where_data_come_from.gif)

In [ ]:
def simulate_reading(n=40, intercept=0.5, slope=0.8, sigma=1.0, seed=226):
    """Simulate n texts: length (hundreds of words) and reading time (minutes)."""
    rng = np.random.default_rng(seed)
    words = rng.uniform(1, 12, size=n).round(1)
    time = intercept + slope * words + rng.normal(0, sigma, size=n)
    return pd.DataFrame({"words": words, "time": time.round(2)})


df = simulate_reading()
df.head()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["words"], df["time"], color=GREY)
ax.set(
    xlabel="Text length (hundreds of words)",
    ylabel="Reading time (minutes)",
    xlim=(0, 12.5),
    ylim=(0, 12),
)
plt.show()

❓ **Look at the plot.** By eye: how much longer does it take to read 100 extra words?
Keep your guess in mind for the next section.

## 2. Guess the line

A line is `time = a + b × words`:

* **a** (intercept): the predicted reading time for a text of length 0,
* **b** (slope): how many extra minutes each extra 100 words take.

For each text, the **residual** is *observed time − predicted time*. A good line has small
residuals. We summarise them with the **residual sum of squares (RSS)**: square every residual
and add them up. The smaller the RSS, the better the line fits.

In [ ]:
def rss(a, b, data=df):
    """Residual sum of squares of the line a + b * words."""
    predicted = a + b * data["words"]
    residuals = data["time"] - predicted
    return (residuals**2).sum()


def plot_guess(a=2.0, b=0.5):
    xs = np.array([0, 12.5])
    predicted = a + b * df["words"]
    fig, ax = plt.subplots()
    ax.vlines(df["words"], predicted, df["time"], color="#A6A6A6", lw=1)  # residuals
    ax.scatter(df["words"], df["time"], color=GREY, zorder=3)
    ax.plot(xs, a + b * xs, color="#404040", lw=2.5)
    ax.set(
        xlabel="Text length (hundreds of words)",
        ylabel="Reading time (minutes)",
        xlim=(0, 12.5),
        ylim=(0, 12),
        title=f"Your line:  RSS = {rss(a, b):.1f}",
    )
    plt.show()


print("RSS of the line a = 2, b = 0.5:", round(rss(2, 0.5), 1))

**Move the sliders** to make the RSS as small as possible. The grey vertical segments are the
residuals.

In [ ]:
from ipywidgets import FloatSlider, interact

interact(
    plot_guess,
    a=FloatSlider(value=2.0, min=-2, max=4, step=0.05, description="intercept a"),
    b=FloatSlider(value=0.5, min=0, max=1.5, step=0.01, description="slope b"),
);

✏️ **Exercise 2.1.** Write down your best guess for `a` and `b` and compute its RSS.

In [ ]:
my_a = ___  # your best intercept
my_b = ___  # your best slope
rss(my_a, my_b)

## 3. Let `statsmodels` find the least-squares line

**Ordinary least squares (OLS)** finds the values of `a` and `b` with the smallest possible RSS.
In `statsmodels` we describe the model with a *formula*: `"time ~ words"` reads
"time explained by words". The intercept is added automatically.

In [ ]:
fit = smf.ols("time ~ words", data=df).fit()
fit.summary()

That is a big table! For now, only look at the **`coef`** column:

* `Intercept` is the estimate of **a**,
* `words` is the estimate of **b**.

The other columns (`std err`, `t`, `P>|t|`, `[0.025, 0.975]`) describe **uncertainty**: they
are the topic of Notebook 2. You can also get just the estimates:

In [ ]:
fit.params

✏️ **Exercise 3.1.** Compute the RSS of the least-squares line with our `rss()` function.
Is it smaller than the RSS of your best guess?

In [ ]:
a_hat = fit.params["___"]
b_hat = fit.params["___"]
rss(a_hat, b_hat)

✏️ **Exercise 3.2.** Compare the estimates with the true values (a = 0.5, b = 0.8).
Are they exactly equal? Why not?

✏️ **Your answer:** …

### Same line, another road: maximum likelihood

Maximum likelihood looks for the line (and σ) under which the observed data are **most
probable**, assuming Normal errors. We can find it with a general-purpose optimiser:
just run the cell and compare the numbers with `statsmodels`.

In [ ]:
from scipy import optimize, stats


def negative_log_likelihood(params):
    a, b, log_sigma = params
    predicted = a + b * df["words"]
    return -stats.norm.logpdf(df["time"], predicted, np.exp(log_sigma)).sum()


result = optimize.minimize(negative_log_likelihood, x0=[0.0, 0.0, 0.0])
print("maximum likelihood: a = %.3f, b = %.3f" % (result.x[0], result.x[1]))
print(
    "least squares:      a = %.3f, b = %.3f"
    % (fit.params["Intercept"], fit.params["words"])
)

Same numbers: with Normal errors, **least squares = maximum likelihood**.

## 4. More data, more noise

The function `simulate_reading()` lets us create new datasets. Here we fit the same model to
datasets of different sizes:

In [ ]:
for n in [10, 40, 400, 4000]:
    d = simulate_reading(n=n, seed=1)
    f = smf.ols("time ~ words", data=d).fit()
    print(
        f"n = {n:4d}:  intercept = {f.params['Intercept']:5.2f},  slope = {f.params['words']:.3f}"
    )

✏️ **Exercise 4.1.** Copy the loop above and add `sigma=3` to `simulate_reading(...)` (three
times more noise). What happens to the estimates?

In [ ]:
for n in [10, 40, 400, 4000]:
    d = simulate_reading(n=n, seed=1)  # ✏️ add sigma=3 here
    f = smf.ols("time ~ words", data=d).fit()
    print(f"n = {n:4d}:  intercept = {f.params['Intercept']:5.2f},  slope = {f.params['words']:.3f}")

✏️ **Your answer:** …

## 5. Prediction

Once we have a line, we can predict the reading time of a new text. `fit.predict()` takes a
table with the predictor values:

In [ ]:
new_texts = pd.DataFrame({"words": [3, 6]})  # 300 and 600 words
fit.predict(new_texts)

✏️ **Exercise 5.1.** Predict the reading time of a **1,000-word** text (`words = 10`). Check the
result by hand with `a_hat + b_hat * 10`.

In [ ]:
fit.predict(pd.DataFrame({"words": [___]}))

✏️ **Exercise 5.2.** What does the model predict for a 5,000-word text (`words = 50`)? Would you
trust this prediction? (Hint: look at the range of text lengths in our data.)

In [ ]:
# ✏️ Your code here

✏️ **Your answer:** …

## Summary

* A line has an **intercept** (a), a **slope** (b) and a residual spread **σ**.
* **Least squares** picks the line with the smallest sum of squared residuals.
  With Normal errors this is also the **maximum likelihood** line.
* `smf.ols("y ~ x", data=df).fit()` fits it; the `coef` column holds the estimates.
* Estimates are close to the truth but never exact; more data → closer.

➡️ Next: **how sure are we?** (Notebook 2)